## NHANES Dataset Analysis

This notebook extends `nn_ns_2.ipynb` with two improvements:
1. **StratifiedKFold** — ensures each fold preserves the class balance (34% positives).
2. **Refit on full training set** — after CV determines the optimal number of epochs, a final model is retrained on 100% of the training data for that many epochs.

In [9]:
import sys
sys.path.append("..")

import os
import pandas as pd
import random
import numpy as np
import torch
from pprint import pprint

from data import normalize_value
from data_real import load_data as load_data_real
from train import train_conformance_inference
from train_bc import cv_loop_bc
from metrics import compute_classification_metrics
from utils import get_device

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, roc_auc_score, log_loss
)

device = get_device()

random.seed(0)
torch.manual_seed(0)
np.random.seed(0)

In [10]:
n_folds = 5
n_epochs = 300
batch_size = 16
learning_rate = 0.001
weight_decay = 0.01
dropout = 0.2
hidden_sizes = [32, 16, 8]
patience_classification = 10
min_delta_classification = 1e-4
standardize = True

In [11]:
def evaluate_on_test(model, test_data):
    """Evalúa el modelo en el conjunto de test y retorna métricas detalladas"""
    model.eval()
    with torch.no_grad():
        y_pred_proba = model(
            torch.from_numpy(test_data['x']).to(device)
        ).cpu().detach().numpy()

    y_pred_bin = (y_pred_proba >= 0.5).astype(int)
    y_true = np.ravel(test_data['y'])

    accuracy = accuracy_score(y_true, y_pred_bin)
    auc_score = roc_auc_score(y_true, y_pred_proba)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average='weighted'
    )
    cm = confusion_matrix(y_true, y_pred_bin)
    ce = log_loss(y_true, y_pred_proba)

    return {
        'accuracy': accuracy,
        'auc': auc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'ce': ce,
        'cm': cm,
    }

In [ ]:
comb_1 = ['RIDAGEYR', 'RIAGENDR']
comb_2 = ['BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
comb_3 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS']
comb_4 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR']
comb_5 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGH']
comb_6 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU']
comb_7 = ['RIDAGEYR', 'RIAGENDR', 'BMXHT', 'BMXWT', 'BMXBMI', 'BMXWAIST', 'BPXDI1', 'BPXSY1', 'BPXPLS', 'LBXSCH', 'LBXSTR', 'LBXGLU', 'LBXGH']
combinations = [comb_1, comb_2, comb_3, comb_4, comb_5, comb_6, comb_7]

# Load pre-generated splits (run generate_splits.py once to create this file)
splits_data = np.load("../data/splits.npz")
train_seqn = splits_data['train_seqn']
test_seqn  = splits_data['test_seqn']

results_list = []

for i, comb in enumerate(combinations):
    print(f"[{i+1}/{len(combinations)}] Procesando combinación {i}: {len(comb)} features")

    file_real = os.path.join("../data", "nhanes_with_metabolic_syndrome_adults.csv")
    data = load_data_real(file_real, comb, 'metabolic_syndrome', 'WTMEC2YR')

    # Map pre-generated SEQN splits to positional indices in this combination's data
    seqn = data['seqn']
    train_split = np.where(np.isin(seqn, train_seqn))[0]
    test_split  = np.where(np.isin(seqn, test_seqn))[0]

    # Build fold splits as indices into the train subset
    train_seqn_local = seqn[train_split]
    fold_splits = []
    for fold_idx in range(n_folds):
        ft = splits_data[f'fold_{fold_idx}_train']
        fv = splits_data[f'fold_{fold_idx}_val']
        fold_splits.append((
            np.where(np.isin(train_seqn_local, ft))[0],
            np.where(np.isin(train_seqn_local, fv))[0],
        ))

    train_data = {
        'seqn': data['seqn'][train_split],
        'x': data['x'][train_split],
        'y': data['y'][train_split],
        'w': data['w'][train_split],
        'z': data['z'][train_split]
    }

    test_data = {
        'seqn': data['seqn'][test_split],
        'x': data['x'][test_split],
        'o': data['x'][test_split],
        'y': data['y'][test_split],
        'w': data['w'][test_split],
        'z': data['z'][test_split]
    }

    if standardize:
        scaler = StandardScaler()
        train_data['x'] = scaler.fit_transform(train_data['x'])
        test_data['x'] = scaler.transform(test_data['x'])

    # Class imbalance correction
    y_train = train_data['y'].ravel()
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model, train_metrics = cv_loop_bc(
        data=train_data,
        splits=fold_splits,
        n_epochs=n_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        patience=patience_classification,
        min_delta=min_delta_classification,
        dropout=dropout,
        hidden_sizes=hidden_sizes,
        refit=False,
        pos_weight=float(pos_weight)  # Reentrenar modelo final sobre todo el training set
    )

    test_metrics = evaluate_on_test(model, test_data)

    result_row = {
        'Combination': i,
        'N_Features': len(comb),
        'Features': ', '.join(comb),
        'Train_AUC': float(train_metrics['auc']),
        'Train_Accuracy': float(train_metrics['accuracy']),
        'Train_Precision': float(train_metrics['precision']),
        'Train_Recall': float(train_metrics['recall']),
        'Train_F1': float(train_metrics['f1']),
        'Train_CE': float(train_metrics['ce']),
        'Test_AUC': float(test_metrics['auc']),
        'Test_Accuracy': float(test_metrics['accuracy']),
        'Test_Precision': float(test_metrics['precision']),
        'Test_Recall': float(test_metrics['recall']),
        'Test_F1': float(test_metrics['f1']),
        'Test_CE': float(test_metrics['ce'])
    }
    results_list.append(result_row)

    print(f"  Train - AUC: {train_metrics['auc']:.4f}, Acc: {train_metrics['accuracy']:.4f}, F1: {train_metrics['f1']:.4f}, CE: {train_metrics['ce']:.4f}")
    print(f"  Test  - AUC: {test_metrics['auc']:.4f}, Acc: {test_metrics['accuracy']:.4f}, F1: {test_metrics['f1']:.4f}, CE: {test_metrics['ce']:.4f}")
    print()

results_df = pd.DataFrame(results_list)

print("\n" + "="*120)
print("RESULTADOS COMPARATIVOS")
print("="*120)
display(results_df[[
    'Combination', 'N_Features',
    'Test_AUC', 'Test_Accuracy', 
    'Test_Recall', 'Test_Precision', 'Test_F1', 
    'Test_CE', 'CI_Correct'
]].style.format({
    'Test_AUC': '{:.4f}', 'Test_Accuracy': '{:.4f}',
    'Test_Recall': '{:.4f}', 'Test_Precision': '{:.4f}',
    'Test_F1': '{:.4f}', 'Test_CE': '{:.4f}',
    'CI_Correct': '{:.4f}'
}).background_gradient(subset=['Test_AUC'], cmap='RdYlGn'))

results_df.to_csv('comparison_results_nn_3.csv', index=False)
print(f"\nResultados guardados en 'comparison_results_nn_3.csv'")


[1/7] Procesando combinación 0: 2 features
  Early stopping at epoch 21
  Early stopping at epoch 12
  Early stopping at epoch 15
  Early stopping at epoch 12
  Early stopping at epoch 33
  Train - AUC: 0.6954, Acc: 0.6095, F1: 0.5670, CE: 0.9509
  Test  - AUC: 0.6874, Acc: 0.5753, F1: 0.5785, CE: 0.6468

[2/7] Procesando combinación 1: 7 features
  Early stopping at epoch 42
  Early stopping at epoch 37
  Early stopping at epoch 40
  Early stopping at epoch 41
  Early stopping at epoch 23
  Train - AUC: 0.8674, Acc: 0.7682, F1: 0.6721, CE: 1.0990
  Test  - AUC: 0.8634, Acc: 0.7706, F1: 0.7761, CE: 0.4606

[3/7] Procesando combinación 2: 9 features
  Early stopping at epoch 29
  Early stopping at epoch 21
  Early stopping at epoch 39
  Early stopping at epoch 35
  Early stopping at epoch 23
  Train - AUC: 0.8730, Acc: 0.7659, F1: 0.6821, CE: 0.9153
  Test  - AUC: 0.8680, Acc: 0.7480, F1: 0.7552, CE: 0.4618

[4/7] Procesando combinación 3: 11 features
  Early stopping at epoch 30
  Earl

,Combination,N_Features,Train_AUC,Train_Accuracy,Train_F1,Train_CE,Test_AUC,Test_Accuracy,Test_F1,Test_CE
0,0,2,0.6954,0.6095,0.5670,0.9509,0.6874,0.5753,0.5785,0.6468
1,1,7,0.8674,0.7682,0.6721,1.0990,0.8634,0.7706,0.7761,0.4606
2,2,9,0.8730,0.7659,0.6821,0.9153,0.8680,0.7480,0.7552,0.4618
3,3,11,0.9315,0.8459,0.7660,0.7419,0.9375,0.8584,0.8605,0.3323
4,4,12,0.9350,0.8488,0.7708,0.7320,0.9372,0.8530,0.8556,0.3498
5,5,12,0.9540,0.8758,0.8031,0.6165,0.9526,0.8677,0.8702,0.3117
6,6,13,0.9551,0.8748,0.8034,0.6535,0.9532,0.8685,0.8708,0.2897



Resultados guardados en 'comparison_results_nn_3.csv'
